<a href="https://colab.research.google.com/github/yeonji200522-oss/DeepLearning2026-2/blob/main/%EB%94%A5%EA%B8%B0_%ED%94%8C%EC%A0%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/maygodwithu/Class2026_DL.git
%cd Class2026_DL/project1

Cloning into 'Class2026_DL'...
remote: Enumerating objects: 10314, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 10314 (delta 9), reused 13 (delta 4), pack-reused 10296 (from 1)
Receiving objects: 100% (10314/10314), 64.55 MiB | 21.52 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/Class2026_DL/project1


In [2]:
from google.colab import files
from pathlib import Path
import io, zipfile, shutil, tempfile

# 기존 학습 코드가 있는 폴더 찾기
TARGET = Path(WORK) if "WORK" in globals() else Path.cwd()
if not (TARGET / "train.py").exists():
    TARGET = Path("/content/Class2026_DL/project1")

assert (TARGET / "train.py").exists(), f"학습 폴더를 확인하세요: {TARGET}"

uploaded = files.upload()
archives = [data for name, data in uploaded.items()
            if name.lower().endswith(".zip")]
assert len(archives) == 1, "ZIP 파일 하나만 선택하세요."

# 기존 코드는 백업하고, 파일 3개만 교체
backup = Path(tempfile.mkdtemp(prefix="cnn_backup_", dir="/content"))
names = ["model.py", "dataset.py", "train.py"]

with zipfile.ZipFile(io.BytesIO(archives[0])) as archive:
    sources = {name: archive.read(f"project1/{name}") for name in names}

for name, source in sources.items():
    if (TARGET / name).exists():
        shutil.copy2(TARGET / name, backup / name)
    (TARGET / name).write_bytes(source)
    print("교체 완료:", TARGET / name)

print("기존 코드 백업:", backup)

Saving cnn_project.zip to cnn_project.zip
교체 완료: /content/Class2026_DL/project1/model.py
교체 완료: /content/Class2026_DL/project1/dataset.py
교체 완료: /content/Class2026_DL/project1/train.py
기존 코드 백업: /content/cnn_backup_0q320zt5


In [11]:
%cd /content/Class2026_DL/project1

from pathlib import Path
from urllib.request import urlopen

base = "https://raw.githubusercontent.com/yeonji200522-oss/DeepLearning2026-2/main/project1/"
names = ["model.py", "dataset.py", "train.py"]

# 모두 내려받은 뒤 기존 파일 교체
sources = {name: urlopen(base + name).read() for name in names}

for name, source in sources.items():
    Path(name).write_bytes(source)
    print("교체 완료:", name)

/content/Class2026_DL/project1
교체 완료: model.py
교체 완료: dataset.py
교체 완료: train.py


In [12]:
!python train.py

Device: cuda; model: deep; run: /content/Class2026_DL/project1/runs/refined_seed42
Parameters + floating buffers: 1,878,437 / 2,000,000
crop_label: counts [1267, 1496, 5237], loss weights [1.243703007698059, 1.1445616483688354, 0.6117355823516846]
health_label: counts [1394, 6606], loss weights [1.2657313346862793, 0.7342686653137207]
Loss weights: crop 1.25, health 1.0; augmentation gentle
Train 8000; valid 2284; AMP True
[01/120] loss 1.6222 crop F1 0.6954 health F1 0.7352 score 0.7153 (raw) lr 0.0002 41.8s aug gentle
  Saved best_model.pt (7.21 MiB)
[02/120] loss 1.2873 crop F1 0.7755 health F1 0.8358 score 0.8057 (ema) lr 0.0004 39.0s aug gentle
  Saved best_model.pt (7.21 MiB)
[03/120] loss 0.9876 crop F1 0.8255 health F1 0.8952 score 0.8604 (ema) lr 0.0006 39.7s aug gentle
  Saved best_model.pt (7.21 MiB)
[04/120] loss 0.9043 crop F1 0.8623 health F1 0.9281 score 0.8952 (ema) lr 0.0008 40.8s aug gentle
  Saved best_model.pt (7.21 MiB)
[05/120] loss 0.8274 crop F1 0.8520 health F1

In [13]:
!python check_submission.py \
    --model runs/refined_seed42//model.py \
    --weights runs/refined_seed42//best_model.pt \
    --data_dir student_data_128

 제출 전 점검 (채점 서버와 같은 방식, CPU)
[통과] 파일 크기: model.py 4.6KB, best_model.pt 7.21MB (제한 10MB)
[통과] import: torch, torchvision, 파이썬 기본 라이브러리만 사용
[통과] model.py 에 MultiTaskCNN 클래스가 있음
[통과] MultiTaskCNN() 을 인자 없이 만들 수 있음
[통과] 모델 크기: 파라미터 1,874,597개 + 버퍼 3,840개 = 1,878,437개 (제한 2,000,000개)
[통과] best_model.pt 를 model.py 구조에 맞게 불러옴
[통과] 출력 형식: (B, 3), (B, 2)  (valid 2284장 예측 완료)
------------------------------------------------------------
 valid 점수 (채점 서버와 같은 전처리)
   작물 종류 Macro-F1 : 0.9736
   건강 상태 Macro-F1 : 0.9908
   평균 Macro-F1      : 0.9822
 모든 검사 통과! 리더보드에 model.py 와 best_model.pt 를 제출하세요.
 (test 점수는 valid 점수와 조금 다를 수 있습니다)


In [9]:
!python check_submission.py                                   # model.py, best_model.pt 가 project1 폴더에 있을 때
!python check_submission.py --model my_model.py --weights my.pt   # 다른 파일을 검사할 때

 제출 전 점검 (채점 서버와 같은 방식, CPU)
[실패] best_model.pt 파일이 없습니다

=> 위 문제를 고친 뒤 다시 실행하세요. 이대로 제출하면 채점에 실패합니다.
 제출 전 점검 (채점 서버와 같은 방식, CPU)
[실패] my_model.py 파일이 없습니다

=> 위 문제를 고친 뒤 다시 실행하세요. 이대로 제출하면 채점에 실패합니다.
